# Day 03 — Stability analysis: cross validation, bootstrap, leakage

**Slide tham khảo chi tiết:** [day03_reference_pack.zip](_static/slides/day03_reference_pack.zip)

## Mục tiêu bài học

- chạy 5-fold cross validation cho mô hình tốt nhất
- ước lượng bootstrap CI cho AUC
- hiểu leakage là gì qua một ví dụ cố ý làm sai
- viết được nhận xét về độ ổn định của mô hình

## Nội dung

Trong báo cáo nghiên cứu, một con số AUC đẹp chưa đủ. Cần trả lời thêm: kết quả có ổn định không, dao động bao nhiêu và có nguy cơ đánh giá lạc quan hay không.

## Sản phẩm sau bài học

- bảng CV scores
- bootstrap mean AUC và khoảng tin cậy
- ví dụ so sánh đúng và sai do leakage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
plt.rcParams["figure.figsize"] = (6,4)

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"


def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out = Path(rel_path).name
    urllib.request.urlretrieve(url, out)
    return Path(out)


def load_csv(rel_path: str) -> pd.DataFrame:
    p = Path(rel_path)
    if p.exists():
        return pd.read_csv(p)
    try:
        p2 = download_from_github(rel_path)
        return pd.read_csv(p2)
    except Exception as e:
        raise FileNotFoundError(f"Không tìm thấy {rel_path}. Hãy chỉnh đúng GITHUB_USER/REPO_NAME hoặc upload file thủ công.") from e


df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")

In [ ]:

feature_set = "ring1"
cols = [c for c in df.columns if c.startswith(feature_set + "_")]
X = df[cols]
y = df["egfr_mutation"].astype(int)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])


## 1. Cross validation AUC

In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
cv_scores


In [ ]:

cv_summary = pd.DataFrame({
    "fold": range(1, len(cv_scores)+1),
    "auc": cv_scores
})
cv_summary.loc[len(cv_summary)] = ["mean", cv_scores.mean()]
cv_summary


## 2. Bootstrap CI cho AUC

Ở đây dùng cách bootstrap đơn giản để nhìn mức dao động của AUC qua nhiều lần lấy mẫu lại.

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe.fit(X_train, y_train)
base_pred = pipe.predict_proba(X_test)[:, 1]
base_auc = roc_auc_score(y_test, base_pred)
base_auc


In [ ]:

rng = np.random.default_rng(42)
boot_aucs = []
y_test_np = y_test.to_numpy()
base_pred_np = np.asarray(base_pred)
for _ in range(300):
    idx = rng.integers(0, len(y_test_np), len(y_test_np))
    y_b = y_test_np[idx]
    p_b = base_pred_np[idx]
    if len(np.unique(y_b)) < 2:
        continue
    boot_aucs.append(roc_auc_score(y_b, p_b))

boot_aucs = np.array(boot_aucs)
ci_low, ci_high = np.quantile(boot_aucs, [0.025, 0.975])
(base_auc, boot_aucs.mean(), ci_low, ci_high)


In [ ]:

out_dir = Path("outputs/day03")
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots()
ax.hist(boot_aucs, bins=20)
ax.axvline(ci_low, color="red", linestyle="--", label=f"2.5%={ci_low:.3f}")
ax.axvline(ci_high, color="red", linestyle="--", label=f"97.5%={ci_high:.3f}")
ax.axvline(base_auc, color="black", linestyle="-", label=f"base AUC={base_auc:.3f}")
ax.set_title("Bootstrap distribution of AUC")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(out_dir / "bootstrap_auc_hist.png", dpi=150)
plt.show()


## 3. Minh họa leakage

Ví dụ dưới đây cố ý làm sai: scale toàn bộ dữ liệu trước rồi mới CV. Khi đó điểm số có thể lạc quan hơn thực tế.

In [ ]:

from sklearn.preprocessing import StandardScaler

X_scaled_wrong = StandardScaler().fit_transform(X)
wrong_scores = cross_val_score(LogisticRegression(max_iter=2000), X_scaled_wrong, y, cv=cv, scoring="roc_auc")

pd.DataFrame({
    "proper_pipeline_cv": cv_scores,
    "leaky_cv": wrong_scores
})


## 4. Cách đọc kết quả ổn định

Khi đọc Day 03, không chỉ nhìn mean AUC. Cần nhìn thêm:

- độ phân tán giữa các fold
- độ rộng của bootstrap CI
- khoảng chênh giữa cách làm đúng và cách làm sai do leakage

Nếu CV dao động quá lớn hoặc bootstrap CI quá rộng, thì chưa nên phát biểu quá mạnh về mô hình.

In [ ]:

cv_summary.to_csv(out_dir / "cv_summary.csv", index=False)
pd.DataFrame({"boot_auc": boot_aucs}).to_csv(out_dir / "bootstrap_aucs.csv", index=False)
pd.DataFrame({"proper_pipeline_cv": cv_scores, "leaky_cv": wrong_scores}).to_csv(out_dir / "leakage_demo.csv", index=False)
list(out_dir.iterdir())


## 5. Tự kiểm tra

1. Vì sao Day 02 chưa đủ để kết luận mô hình tốt?  
2. Bootstrap CI đang trả lời câu hỏi gì?  
3. Leakage trong ví dụ trên xảy ra ở bước nào?  
4. Nếu mean AUC cao nhưng CI rất rộng thì phải diễn giải thế nào?

## Nối sang buổi sau

Day 04 sẽ dùng mô hình đã hiểu về hiệu năng và độ ổn định để nối sang subset NGS, tính delta P theo pathway.